# (4) Human in the Loop

이 단원에서는 에이전트의 도구 실행 직전 사람의 개입 및 승인 단계를 거치는 **Human-in-the-loop (HITL)** 설계 기법을 배웁니다.

공식 문서: [LangChain Human-in-the-Loop 가이드](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)

In [ ]:
# 1. 환경 변수 로드 및 초기화
import sys
import os
import re
from dotenv import load_dotenv

# ⚠️ 주피터 실행 디렉토리(notebooks/)와 프로젝트 루트(agent-harness-lab/) 경로 싱크 정합
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print(f"🔄 작업 디렉토리를 프로젝트 루트('{os.getcwd()}')로 전환 완료.\n")

# LangSmith API Key Forbidden 경고 차단
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# 프로젝트 루트 경로 기준 src 추가 및 .env 수동 로드
sys.path.append(os.path.abspath("src"))
load_dotenv(override=True)

from utils.llm import get_llm
# AAWS 연동 텍스트 정규화 유틸리티 임포트
from utils.message_utils import normalize_content
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from app.tools import web_search, file_read, file_writer  # 🏭 우리의 챗봇 범용 3대 무기 임포트!

# (A) 자체 제작 파일 도구 셋업
tools = [web_search, file_read, file_writer]

print("tool names :", [t.name for t in tools])

File tool names : ['web_search', 'file_read', 'file_writer']


### Human-in-the-loop 정책 설정 (강의용)

| 설정값 | 인터럽트 | approve | edit | reject |
| :--- | :--- | :--- | :--- | :--- |
| `False` | ❌ 없음 | 자동 실행 | 자동 실행 | 자동 실행 |
| `True` | ✅ 있음 | 가능 | 가능 | 가능 |
| `{"allowed_decisions": ["approve", "reject"]}` | ✅ 있음 | 가능 | ❌ 불가 | 가능 |

In [ ]:
# -------------------------------------------------
# Human-in-the-loop 정책 설정 (자체 제작 도구 타겟)
# -------------------------------------------------
hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        # 검색은 사람이 검토 + 수정 가능
        web_search.name : True,

        # 파일 쓰기는 승인/거절만 가능 (edit 금지)
        "file_writer": {
            "allowed_decisions": ["approve", "reject"]
        },

        # 파일 읽기는 안전하므로 자동 실행
        "file_read": False,
    },
    description_prefix="🛑 Tool execution pending human review",
)

# Vertex AI 오탐지 방지 및 명확한 OpenAI 기동을 위해 openai:gpt-4o를 지정합니다.
agent = create_agent(
    model="openai:gpt-4o",
    tools=tools,
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

### 1단계: 검색 도구 호출 직전 인터럽트(Interrupt) 발생 검증

In [43]:
config = {"configurable": {"thread_id": "hitl_demo_1"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "2026 트럼프 관세에 대해 최신 업데이트 사항을 찾아봐"
            }
        ]
    },
    config=config
)

# ✅ 여기서 멈추면 __interrupt__가 생깁니다.
print("HAS_INTERRUPT =", "__interrupt__" in result)

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_FeQTLI5Tpd2TAzANzX2cIxI7", 'type': 'invalid_request_error', 'param': 'messages.[2].role', 'code': None}}

In [44]:
# 사람이 리뷰해야 할 액션(툴 호출 요청) 확인
interrupts = result.get("__interrupt__", [])
print("HAS_INTERRUPT =", bool(interrupts))

if interrupts:
    req = interrupts[0].value

    print("\n=== ACTION REQUESTS ===")
    for i, ar in enumerate(req.get("action_requests", [])):
        tool_name = ar.get("name")
        tool_args = ar.get("args", ar.get("arguments"))  # ✅ args 우선, 없으면 arguments
        desc = ar.get("description")

        print(f"\n[#{i}] tool =", tool_name)
        print("args =", tool_args)
        print("desc =", desc)

    print("\n=== REVIEW CONFIGS ===")
    print(req.get("review_configs"))

HAS_INTERRUPT = True

=== ACTION REQUESTS ===

[#0] tool = web_search
args = {'query': 'Trump tariffs latest update October 2023'}
desc = 🛑 Tool execution pending human review

Tool: web_search
Args: {'query': 'Trump tariffs latest update October 2023'}

=== REVIEW CONFIGS ===
[{'action_name': 'web_search', 'allowed_decisions': ['approve', 'edit', 'reject']}]


### 2단계: 승인 (APPROVE)

사람이 승인 의사결정을 전달하여 에이전트를 재기동(Resume)시킵니다.

In [45]:
from langgraph.types import Command

result2 = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)

# 최종 assistant 메시지 출력
msgs = result2.get("messages", [])
print(msgs[-1].content if msgs else "(no messages)")

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_FeQTLI5Tpd2TAzANzX2cIxI7", 'type': 'invalid_request_error', 'param': 'messages.[2].role', 'code': None}}

### 3단계: 수정 및 실행 (EDIT)

사람이 에이전트가 올린 도구 인자값을 직접 교정해서 실행을 위임합니다.

In [ ]:
config = {"configurable": {"thread_id": "hitl_demo_2"}}

result3 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "트럼프 관세에 대해 최신 업데이트 사항을 찾아봐"
            }
        ]
    },
    config=config
)
print("HAS_INTERRUPT =", "__interrupt__" in result3)

In [ ]:
result4 = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "web_search",
                        "args": {
                            "query": "트럼프 관세 최신 업데이트 일본 반응"
                        },
                    },
                }
            ]
        }
    ),
    config=config
)

msgs = result4.get("messages", [])
print(msgs[-1].content if msgs else "(no messages)")

### 4단계: 거절 (REJECT)

파일 쓰기 작업(`file_writer`)을 강제로 거절하여 보안 차단 피드백을 전달합니다.

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "hitl_write_reject_demo_1"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "아래 내용을 './sandbox/memo.txt' 파일로 저장해줘.\n\n"
                    "- HITL은 tool 실행 전에 멈춘다\n"
                    "- approve/edit/reject로 사람이 개입한다\n"
                )
            }
        ]
    },
    config=config
)

print("HAS_INTERRUPT =", "__interrupt__" in result)
print(result.get("__interrupt__"))

In [ ]:
# (교육용) 어떤 tool call이 멈췄는지 보기
if "__interrupt__" in result:
    req = result["__interrupt__"][0].value
    ar = req["action_requests"][0]
    print("\nTool =", ar.get("name"))
    print("Args =", ar.get("args", ar.get("arguments")))
    print("Allowed =", req["review_configs"][0].get("allowed_decisions"))

In [ ]:
result2 = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": (
                        "파일 쓰기 작업은 승인되지 않았습니다. (실습 목적상 file_writer 거절)\n"
                        "대신, 내용을 채팅으로만 출력하고 저장이 필요하면 사용자에게 승인(approve)을 요청하세요."
                    )
                }
            ]
        }
    ),
    config=config
)

msgs = result2.get("messages", [])
print(msgs[-1].content if msgs else "(no messages)")